In [1]:
!unzip -q processed_csv.zip

In [2]:
import os
import glob

# Let's inspect where the CSV files were extracted to find the correct directory
csv_files = glob.glob('**/*.csv', recursive=True)
if csv_files:
    print(f"Found {len(csv_files)} CSV files.")
    # Get the directory of the first found CSV file
    target_folder = os.path.dirname(csv_files[0])
    if target_folder == '':
        target_folder = '.'
    print(f"Correct folder containing CSV files is: '{target_folder}'")
else:
    print("No CSV files found in the environment. Please verify that 'processed_csv.zip' was successfully unzipped.")

Found 4504 CSV files.
Correct folder containing CSV files is: 'processed_csv'


In [3]:
"""1D CNN fall detection on processed SisFall CSVs (50 Hz, 6 channels).

Files: adl.D10_SA02_R01.csv / fall.F05_SA06_R01.csv
Columns: timestamp(ms), accX, accY, accZ (g), gyrX, gyrY, gyrZ (deg/s)

Usage: python train_fall_cnn.py /path/to/processed_csv
"""
import sys, glob, os
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import confusion_matrix

FS = 50
WIN = 100            # 2 s at 50 Hz
STRIDE = 25          # 0.5 s hop for ADL windows
PEAK_MARGIN = 30     # fall window must contain peak with >=30 samples of context on each side
FALL_HOP = 5         # shift of fall windows around the impact (augmentation)
COLS = ["accX", "accY", "accZ", "gyrX", "gyrY", "gyrZ"]
SEED = 42


# ---------------------------------------------------------------- data
def make_windows(path):
    """Return (windows, label) for one recording."""
    name = os.path.basename(path)
    is_fall = name.startswith("fall")
    x = pd.read_csv(path)[COLS].values.astype("float32")
    if len(x) < WIN:
        return np.empty((0, WIN, 6), "float32")

    if not is_fall:
        starts = range(0, len(x) - WIN + 1, STRIDE)
    else:
        # Only windows around the impact are labelled fall; the rest of a
        # fall recording (standing before / lying after) is ambiguous -> dropped.
        mag = np.linalg.norm(x[:, :3], axis=1)
        pk = int(np.argmax(mag))
        lo = max(0, pk - WIN + PEAK_MARGIN)
        hi = min(len(x) - WIN, pk - PEAK_MARGIN)
        starts = range(lo, hi + 1, FALL_HOP)
    w = [x[s:s + WIN] for s in starts]
    return np.stack(w) if w else np.empty((0, WIN, 6), "float32")


def load_dataset(folder):
    X, y, subj = [], [], []
    files = sorted(glob.glob(os.path.join(folder, "*.csv")))
    if not files:
        # Try recursive search if not found in root of the folder
        files = sorted(glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True))

    print(f"Found {len(files)} CSV files inside '{folder}'.")

    for p in files:
        name = os.path.basename(p)
        w = make_windows(p)
        if len(w) == 0:
            continue
        X.append(w)
        y += [1 if name.startswith("fall") else 0] * len(w)
        subj += [name.split("_")[1]] * len(w)          # e.g. SA02

    if not X:
        raise ValueError(f"No valid segments of length >= {WIN} could be extracted from any CSV files in '{folder}'.")

    return np.concatenate(X), np.array(y, "int32"), np.array(subj)


def subject_split(subj, seed=SEED):
    """70/15/15 split by subject, done separately for SA (young) and SE (elderly)."""
    rng = np.random.RandomState(seed)
    tr, va, te = [], [], []
    for grp in ("SA", "SE"):
        ids = sorted({s for s in subj if s.startswith(grp)})
        rng.shuffle(ids)
        n = len(ids)
        n_tr, n_va = int(round(0.70 * n)), int(round(0.15 * n))
        tr += ids[:n_tr]; va += ids[n_tr:n_tr + n_va]; te += ids[n_tr + n_va:]
    m = lambda ids: np.isin(subj, ids)
    return m(tr), m(va), m(te), (tr, va, te)


# ---------------------------------------------------------------- model
def build_model(win=WIN):
    import tensorflow as tf
    from tensorflow.keras import layers, models
    m = models.Sequential([
        layers.Input((win, 6)),
        layers.Conv1D(16, 5, strides=2, padding="same"),
        layers.BatchNormalization(), layers.ReLU(),
        layers.MaxPooling1D(2),
        layers.Conv1D(32, 5, padding="same"),
        layers.BatchNormalization(), layers.ReLU(),
        layers.MaxPooling1D(2),
        layers.Conv1D(32, 3, padding="same"),
        layers.BatchNormalization(), layers.ReLU(),
        layers.GlobalAveragePooling1D(),
        layers.Dropout(0.3),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(tf.keras.optimizers.Adam(1e-3), "binary_crossentropy",
              metrics=["accuracy",
                       tf.keras.metrics.Recall(name="recall"),
                       tf.keras.metrics.Precision(name="precision")])
    return m


# ---------------------------------------------------------------- main
def main(folder):
    import tensorflow as tf
    tf.keras.utils.set_random_seed(SEED)

    X, y, subj = load_dataset(folder)
    print("windows:", X.shape, "falls:", int(y.sum()), "ADL:", int((y == 0).sum()))

    mtr, mva, mte, (tr_ids, va_ids, te_ids) = subject_split(subj)
    print("train/val/test subjects:", len(tr_ids), len(va_ids), len(te_ids))
    Xtr, ytr, Xva, yva, Xte, yte = X[mtr], y[mtr], X[mva], y[mva], X[mte], y[mte]
    print("windows  train/val/test:", len(ytr), len(yva), len(yte))

    # channel-wise z-score, statistics from TRAIN only
    mean = Xtr.mean(axis=(0, 1)); std = Xtr.std(axis=(0, 1)) + 1e-8
    np.savez("norm_stats.npz", mean=mean, std=std)
    print("mean:", mean.round(3), "\nstd:", std.round(3))
    norm = lambda a: ((a - mean) / std).astype("float32")
    Xtr, Xva, Xte = norm(Xtr), norm(Xva), norm(Xte)

    n0, n1 = (ytr == 0).sum(), (ytr == 1).sum()
    cw = {0: len(ytr) / (2 * n0), 1: len(ytr) / (2 * n1)}

    model = build_model()
    model.summary()
    cb = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=12,
                                         restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                             patience=5, min_lr=1e-5),
    ]
    model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=80, batch_size=128,
              class_weight=cw, callbacks=cb, verbose=2)

    # ---- evaluation (threshold 0.5)
    p = model.predict(Xte, verbose=0).ravel()
    pred = (p >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(yte, pred, labels=[0, 1]).ravel()
    acc = (tp + tn) / len(yte)
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    spec = tn / max(tn + fp, 1); f1 = 2 * prec * rec / max(prec + rec, 1e-9)
    print(f"\nTEST  acc={acc:.4f} precision={prec:.4f} recall={rec:.4f} "
          f"specificity={spec:.4f} F1={f1:.4f}")
    print("confusion matrix [[TN FP],[FN TP]]:\n", np.array([[tn, fp], [fn, tp]]))
    model.save("fall_cnn.keras")

    # ---- TFLite full INT8 + C header
    def rep():
        idx = np.random.RandomState(0).choice(len(Xtr), min(500, len(Xtr)), replace=False)
        for i in idx:
            yield [Xtr[i:i + 1]]

    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset = rep
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type = tf.int8
    conv.inference_output_type = tf.int8
    tfl = conv.convert()
    open("fall_cnn_int8.tflite", "wb").write(tfl)

    # check quantized accuracy on test set
    it = tf.lite.Interpreter(model_content=tfl); it.allocate_tensors()
    i_d, o_d = it.get_input_details()[0], it.get_output_details()[0]
    s_i, z_i = i_d["quantization"]; s_o, z_o = o_d["quantization"]
    qp = []
    for a in Xte:
        q = np.clip(np.round(a / s_i + z_i), -128, 127).astype(np.int8)[None]
        it.set_tensor(i_d["index"], q); it.invoke()
        qp.append((float(it.get_tensor(o_d["index"])[0, 0]) - z_o) * s_o)
    qpred = (np.array(qp) >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(yte, qpred, labels=[0, 1]).ravel()
    print(f"INT8  recall={tp/max(tp+fn,1):.4f} specificity={tn/max(tn+fp,1):.4f}")
    print(f"input scale/zero-point: {s_i}, {z_i}   output: {s_o}, {z_o}")

    with open("model_data.h", "w") as f:
        f.write("// 50 Hz, window=100, channels=6 (ax ay az gx gy gz)\n")
        f.write(f"const float NORM_MEAN[6] = {{{', '.join(f'{v:.6f}f' for v in mean)}}};\n")
        f.write(f"const float NORM_STD[6]  = {{{', '.join(f'{v:.6f}f' for v in std)}}};\n")
        f.write(f"const float INPUT_SCALE = {s_i:.9g}f; const int INPUT_ZERO_POINT = {int(z_i)};\n")
        f.write(f"const float OUTPUT_SCALE = {s_o:.9g}f; const int OUTPUT_ZERO_POINT = {int(z_o)};\n")
        f.write("alignas(8) const unsigned char model_data[] = {\n")
        for i in range(0, len(tfl), 12):
            f.write("  " + ", ".join(f"0x{b:02x}" for b in tfl[i:i + 12]) + ",\n")
        f.write(f"}};\nconst unsigned int model_data_len = {len(tfl)};\n")
    print("wrote model_data.h", len(tfl), "bytes")


if __name__ == "__main__":
    # If run in an interactive notebook (like Colab), ignore connection file parameters
    is_jupyter = any('ipykernel' in arg or 'colab' in arg for arg in sys.argv[0:1] + sys.argv[1:2])

    if is_jupyter or len(sys.argv) <= 1:
        target_dir = "processed_csv"
    else:
        target_dir = sys.argv[1]

    main(target_dir)

Found 4500 CSV files inside 'processed_csv'.
windows: (112468, 100, 6) falls: 16182 ADL: 96286
train/val/test subjects: 26 5 7
windows  train/val/test: 76782 14491 21195
mean: [ 0.008 -0.787 -0.136 -0.337  1.772 -0.088] 
std: [ 0.419  0.61   0.487 46.497 38.354 32.89 ]


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 50, 16)         │           496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 50, 16)         │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 50, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 25, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 25, 32)         │         2,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 25, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 25, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 12, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 12, 32)         │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 12, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 12, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,057 (27.57 KB)

 Trainable params: 6,897 (26.94 KB)

 Non-trainable params: 160 (640.00 B)

Epoch 1/80
600/600 - 23s - 38ms/step - accuracy: 0.9767 - loss: 0.0649 - precision: 0.8616 - recall: 0.9938 - val_accuracy: 0.9886 - val_loss: 0.0429 - val_precision: 0.9458 - val_recall: 0.9743 - learning_rate: 0.0010
Epoch 2/80
600/600 - 2s - 4ms/step - accuracy: 0.9968 - loss: 0.0123 - precision: 0.9813 - recall: 0.9965 - val_accuracy: 0.9930 - val_loss: 0.0368 - val_precision: 0.9762 - val_recall: 0.9738 - learning_rate: 0.0010
Epoch 3/80
600/600 - 2s - 4ms/step - accuracy: 0.9974 - loss: 0.0094 - precision: 0.9848 - recall: 0.9972 - val_accuracy: 0.9928 - val_loss: 0.0348 - val_precision: 0.9743 - val_recall: 0.9743 - learning_rate: 0.0010
Epoch 4/80
600/600 - 2s - 4ms/step - accuracy: 0.9985 - loss: 0.0063 - precision: 0.9909 - recall: 0.9982 - val_accuracy: 0.9955 - val_loss: 0.0358 - val_precision: 0.9935 - val_recall: 0.9743 - learning_rate: 0.0010
Epoch 5/80
600/600 - 2s - 4ms/step - accuracy: 0.9989 - loss: 0.0049 - precision: 0.9941 - recall: 0.9983 - val_accuracy: 0.9925 -

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


INT8  recall=0.9946 specificity=0.9957
input scale/zero-point: 0.2453230619430542, 28   output: 0.00390625, -128
wrote model_data.h 19824 bytes
